Implementation with LangGraph

In [0]:
%pip install langgraph langchain_openai kneed scikit-learn delta-spark


1. Define the State

In [0]:
from typing import TypedDict, List
import pandas as pd

class SalesAgentState(TypedDict):
    target_zips: List[str]
    is_drifted: bool
    processing_log: List[str]
    batch_data: pd.DataFrame


2. The Agentic Nodes

In [0]:
from langgraph.graph import StateGraph, END
from delta.tables import DeltaTable

def prioritize_zips_node(state: SalesAgentState):
    """Calculates Importance and picks the Pareto top 100."""
    # Logic from our previous importance query
    importance_df = spark.table("sales_orders").groupBy("zipcode").agg(
        F.sum(F.when(F.col("type") == "SALES", F.col("value"))).alias("val")
    ).orderBy(F.desc("val")).limit(5) # Example top 5 for speed
    
    zips = [r.zipcode for r in importance_df.collect()]
    return {"target_zips": zips, "processing_log": ["Zips Prioritized"]}

def cluster_and_refine_node(state: SalesAgentState):
    """Runs the Hybrid DBSCAN on the selected batch."""
    zips = state["target_zips"]
    pdf = spark.table("sales_orders").filter(F.col("zipcode").isin(zips)).toPandas()
    
    # ... [Insert the Hybrid DBSCAN + TF-IDF logic here] ...
    # We assign cluster_id and set status to 'PROCESSED'
    pdf['status'] = 'PROCESSED'
    pdf['assignment'] = "Assigned via LangGraph Agent"
    
    return {"batch_data": pdf, "processing_log": ["Spatial Clustering Complete"]}

def manage_table_node(state: SalesAgentState):
    """Handles the Delta Lake Merge (The Librarian)."""
    updates_df = spark.createDataFrame(state["batch_data"])
    master_table = DeltaTable.forName(spark, "sales_orders")
    
    master_table.alias("t").merge(
        updates_df.alias("s"), "t.order_id = s.order_id"
    ).whenMatchedUpdate(set = {
        "status": "s.status",
        "cluster_id": "s.cluster_id",
        "assignment": "s.assignment",
        "last_updated": "current_timestamp()"
    }).execute()
    
    return {"processing_log": ["Delta Table Synchronized"]}

def generate_agenda_node(state: SalesAgentState):
    """Agentic output: Creates the weekly agenda for the CRM."""
    # This could call an LLM to write a summary for the sales rep
    summary = state["batch_data"].groupby('assignment').size().to_dict()
    print(f"Agenda Generated: {summary}")
    return {"processing_log": ["Agendas dispatched to Sales Reps"]}


3. Define the Graph Logic

In [0]:
workflow = StateGraph(SalesAgentState)

# Add Nodes
workflow.add_node("prioritizer", prioritize_zips_node)
workflow.add_node("clusterer", cluster_and_refine_node)
workflow.add_node("librarian", manage_table_node)
workflow.add_node("scheduler", generate_agenda_node)

# Define Flow
workflow.set_entry_point("prioritizer")
workflow.add_edge("prioritizer", "clusterer")
workflow.add_edge("clusterer", "librarian")
workflow.add_edge("librarian", "scheduler")
workflow.add_edge("scheduler", END)

# Compile the Agent
app = workflow.compile()


- How this manages the "Agenda"

The scheduler node is where the "Agentic AI" shines. Instead of just marking a table, it can:
Look at the Cluster: "I see 5 high-value sales leads in Cluster 12 (90210)."
Check the Staff: "Rep Sarah is the closest."
Output: Write a JSON object directly to your Sales CRM API or a Databricks Notification.

- Why LangGraph for this?

Resilience: If the librarian node fails (e.g., a network error during the Merge), LangGraph can retry that specific step without re-running the expensive Clustering step.
Stateful Memory: The agent remembers which Zip Codes it tried to process in previous nodes.
Human-in-the-loop: You can add a "Break" node before manage_table_node that requires a Sales Manager to click "Approve" in a Databricks UI before the agenda is finalized.

#### Just adding a new node in the agent. Will check validation later


In [0]:
from typing import Literal
from langgraph.graph import StateGraph, END

# --- 1. Router Function (The Logic Gate) ---
def check_drift_condition(state: SalesAgentState) -> Literal["recluster", "incremental"]:
    """
    Determines if the Zip Code's spatial density has drifted.
    Reads 'is_drifted' flag calculated in the previous node.
    """
    if state.get("is_drifted", False):
        return "recluster"
    return "incremental"

# --- 2. Node Implementations ---
def monitor_and_detect_drift_node(state: SalesAgentState):
    """Checks new data against existing cluster centroids."""
    # (Insert the PySpark Haversine drift logic from previous response)
    # If any point dist > 1.5km, we set is_drifted = True
    drift_detected = check_pyspark_drift_logic() 
    
    return {
        "is_drifted": drift_detected, 
        "processing_log": [f"Drift check complete. Drift Detected: {drift_detected}"]
    }

def incremental_update_node(state: SalesAgentState):
    """Assigns new points to the NEAREST existing cluster without redrawing boundaries."""
    # Logic: K-Nearest Neighbors to find the closest cluster_id
    return {"processing_log": ["New leads assigned to existing clusters."]}

# --- 3. Build the Stateful Graph ---
workflow = StateGraph(SalesAgentState)

# Add Nodes
workflow.add_node("monitor", monitor_and_detect_drift_node)
workflow.add_node("recluster", cluster_and_refine_node) # Full Hybrid DBSCAN
workflow.add_node("incremental", incremental_update_node) # Lightweight
workflow.add_node("librarian", manage_table_node)
workflow.add_node("scheduler", generate_agenda_node)

# --- 4. Define the Agentic Routing ---
workflow.set_entry_point("monitor")

# Conditional Edge: Decide path based on Drift
workflow.add_conditional_edges(
    "monitor",
    check_drift_condition,
    {
        "recluster": "recluster",
        "incremental": "incremental"
    }
)

# Merge back to the main path
workflow.add_edge("recluster", "librarian")
workflow.add_edge("incremental", "librarian")
workflow.add_edge("librarian", "scheduler")
workflow.add_edge("scheduler", END)

app = workflow.compile()



- Why this is a "Manager" AI Cost Efficiency: By routing to incremental, you avoid running TF-IDF and \(N^{2}\) Distance Matrices for every small change. You only "re-think" the map when it breaks.

- Persistence: LangGraph stores the State at every node. If the Databricks cluster preempts, the agent resumes exactly where it left off (e.g., it won't re-calculate drift if it already determined the Zip was STALE).

- Weekly Agenda Output: The final scheduler node takes the finalized Delta Table updates and formats them into a "Route Plan" for each Sales Rep based on their cluster_id.

#### 1. The Judge Agent Node

In [0]:
def judge_guardrail_node(state: SalesAgentState):
    """
    The Judge: Uses summary statistics to validate if a Re-Cluster is 
    actually required or if the Recluster node is 'overreacting'.
    """
    zip_code = state["target_zips"][0] # Processing one zip context
    
    # 1. Fetch Summary Stats (Cheap Aggregates)
    stats = spark.table("sales_orders").filter(F.col("zipcode") == zip_code).agg(
        F.stddev("latitude").alias("lat_std"),
        F.stddev("longitude").alias("lon_std"),
        F.countDistinct("street_address").alias("unique_streets"),
        F.count("*").alias("total_count")
    ).collect()[0]

    # 2. Heuristic: The 'Jiggle' Test
    # If the standard deviation of location hasn't changed by more than 10%, 
    # the spatial structure is likely the same.
    spatial_variance = stats['lat_std'] + stats['lon_std']
    
    # Logic: Even if the Monitor says 'Drifted', the Judge checks if it's 
    # just a single outlier or a genuine shift in density.
    if spatial_variance < 0.05 and stats['total_count'] < 500:
        decision = "incremental" # Overrule 'recluster'
        reason = "Spatial variance too low to justify full DBSCAN compute."
    else:
        decision = "recluster"
        reason = "Complexity threshold met. Full re-clustering authorized."

    return {
        "judge_decision": decision,
        "processing_log": [f"Judge Decision: {decision} ({reason})"]
    }


#### 
2. Updating the LangGraph Architecture

In [0]:
# --- 1. The Decision Logic ---
def route_after_judge(state: SalesAgentState) -> Literal["recluster", "incremental"]:
    return state["judge_decision"]

# --- 2. Build the Graph ---
workflow = StateGraph(SalesAgentState)

workflow.add_node("monitor", monitor_and_detect_drift_node)
workflow.add_node("judge", judge_guardrail_node) # <--- The Guardrail
workflow.add_node("recluster", cluster_and_refine_node)
workflow.add_node("incremental", incremental_update_node)
workflow.add_node("librarian", manage_table_node)
workflow.add_node("scheduler", generate_agenda_node)

# --- 3. Flow with Guardrail ---
workflow.set_entry_point("monitor")

# Monitor passes to Judge regardless of findings
workflow.add_edge("monitor", "judge")

# Judge decides the final path
workflow.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "recluster": "recluster",
        "incremental": "incremental"
    }
)

workflow.add_edge("recluster", "librarian")
workflow.add_edge("incremental", "librarian")
workflow.add_edge("librarian", "scheduler")
workflow.add_edge("scheduler", END)


3. Why this is more efficient Feature Traditional TriggerAgentic Judge GuardrailCompute CostHigh (\(N^{2}\) comparisons)Low (Single pass aggregation)False PositivesHigh (Every new point triggers drift)Low (Only "Massive Shifts" trigger drift)Pareto AlignmentStaticDynamic (Judge ignores "noise" in low-value Zip Codes)

Real-World Scenario Imagine a new Sales Order is placed on the far edge of a Zip Code. The Monitor sees a distance \(>1.5km\) and panics, flagging for RE-CLUSTER.The Judge looks at the Zip Code and says: "Wait, that's just one order in a rural area. We don't need to rebuild the whole clustering model for the other 400 points."Result: The Judge downgrades the task to INCREMENTAL, saving you minutes of cluster compute time. Would you like me to show you how to add a "Cost Tracker" to the Judge so it can report how much money it saved you by avoiding unnecessary re-clustering?  Creating a public link...HelpfulNot helpfulThank youYour feedback helps Google improve. See our Privacy Policy.Share more feedbackReport a problemClose

In [0]:
class SalesAgentState(TypedDict):
    target_zips: List[str]
    is_drifted: bool
    judge_decision: str
    cost_metrics: dict # Tracking: {'spent': 0.0, 'saved': 0.0}
    processing_log: List[str]
    batch_data: pd.DataFrame


In [0]:
def judge_with_cost_tracker_node(state: SalesAgentState):
    # --- CONFIGURATION (Example Pricing) ---
    COST_PER_ROW_DBSCAN = 0.005  # Heavy: TF-IDF + Matrix Math
    COST_PER_ROW_INC = 0.0002    # Light: Simple Join/KNN
    
    zip_code = state["target_zips"][0]
    row_count = spark.table("sales_orders").filter(F.col("zipcode") == zip_code).count()
    
    # 1. Calculate Potential Costs
    full_recluster_est = row_count * COST_PER_ROW_DBSCAN # system.billing.usage "databricks"
    incremental_est = row_count * COST_PER_ROW_INC
    
    # 2. Re-apply Heuristic Logic (The Jiggle Test)
    # [Insert your previous spatial variance logic here]
    # For this example, let's assume we choose 'incremental'
    decision = "incremental" 
    
    # 3. Track Savings
    savings = 0.0
    actual_cost = incremental_est
    
    if decision == "incremental":
        savings = full_recluster_est - incremental_est
        status_msg = f"Judge SAVED ${savings:.2f} by avoiding full re-cluster."
    else:
        actual_cost = full_recluster_est
        status_msg = f"Judge AUTHORIZED ${actual_cost:.2f} spend for full re-cluster."

    # 4. Update State
    current_metrics = state.get("cost_metrics", {"spent": 0.0, "saved": 0.0})
    current_metrics["spent"] += actual_cost
    current_metrics["saved"] += savings
    
    return {
        "judge_decision": decision,
        "cost_metrics": current_metrics,
        "processing_log": [status_msg]
    }


In [0]:
def generate_agenda_with_roi_node(state: SalesAgentState):
    metrics = state["cost_metrics"]
    total_saved = metrics["saved"]
    total_spent = metrics["spent"]
    roi_percent = (total_saved / (total_spent + 0.001)) * 100
    
    report = f"""
    --- WEEKLY AGENTIC SUMMARY ---
    Total Savings: ${total_saved:.2f}
    Total Spend:   ${total_spent:.2f}
    Efficiency:    {roi_percent:.1f}% more efficient than static clustering.
    ------------------------------
    Assignments have been dispatched to CRM.
    """
    print(report)
    return {"processing_log": [f"Agenda and ROI Report generated."]}


In [0]:
workflow = StateGraph(SalesAgentState)

workflow.add_node("monitor", monitor_and_detect_drift_node)
workflow.add_node("judge", judge_with_cost_tracker_node) # Tracks $
workflow.add_node("recluster", cluster_and_refine_node)
workflow.add_node("incremental", incremental_update_node)
workflow.add_node("librarian", manage_table_node)
workflow.add_node("scheduler", generate_agenda_with_roi_node) # Reports $

# ... (Conditional edges as before) ...
# --- 3. Flow with Guardrail ---
workflow.set_entry_point("monitor")

# Monitor passes to Judge regardless of findings
workflow.add_edge("monitor", "judge")

# Judge decides the final path
workflow.add_conditional_edges(
    "judge",
    route_after_judge,
    {
        "recluster": "recluster",
        "incremental": "incremental"
    }
)

workflow.add_edge("recluster", "librarian")
workflow.add_edge("incremental", "librarian")
workflow.add_edge("librarian", "scheduler")
workflow.add_edge("scheduler", END)